<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

## 🗺️ Dtype Mapping

Map Arrow types to MAX dtypes for proper tensor creation.

| Arrow Type | MAX DType | NumPy dtype |
|------------|-----------|-------------|
| `int8` - `int64` | `DType.int8` - `DType.int64` | `np.int8` - `np.int64` |
| `uint8` - `uint64` | `DType.uint8` - `DType.uint64` | `np.uint8` - `np.uint64` |
| `float32`, `float64` | `DType.float32`, `DType.float64` | `np.float32`, `np.float64` |
| `date32` | `DType.int32` | `np.int32` *(epoch days)* |

In [0]:
#| echo: false
#| output: asis
show_doc(get_numpy_dtype)

---

### get_numpy_dtype

```python

def get_numpy_dtype(
    arrow_type:DataType
)->dtype:


```

*Get NumPy dtype for an Arrow type.*

In [0]:
#| echo: false
#| output: asis
show_doc(get_max_dtype)

---

### get_max_dtype

```python

def get_max_dtype(
    arrow_type:DataType
)->DType:


```

*Get MAX DType for an Arrow type.*

## 🔄 Zero-Copy Bridge Functions

The key insight: PyArrow stores data in **contiguous buffers**. For primitive types without nulls, `arr.to_numpy(zero_copy_only=True)` gives a NumPy view over the **same memory**.

We then use `driver.Tensor.from_numpy()` which maintains the zero-copy property for contiguous arrays.

### Memory Flow

```
┌──────────────┐     zero-copy      ┌──────────────┐     zero-copy      ┌──────────────┐
│  Arrow Array │ ─────────────────► │  NumPy View  │ ─────────────────► │  MAX Tensor  │
│   (buffer)   │                    │  (same mem)  │                    │  (same mem)  │
└──────────────┘                    └──────────────┘                    └──────────────┘
```

### ⚠️ Constraints

- ❌ **No nulls** - Arrow validity bitmaps break zero-copy
- ❌ **Single chunk** - Multiple chunks require combining
- ❌ **GPU requires copy** - Device memory is separate
- ✅ **Primitive types only** - Strings/nested types not supported

In [0]:
#| echo: false
#| output: asis
show_doc(arrow_to_numpy_view)

---

### arrow_to_numpy_view

```python

def arrow_to_numpy_view(
    arr:Union
)->ndarray:


```

*Get a zero-copy NumPy view of an Arrow array.*

Args:
    arr: PyArrow Array or ChunkedArray (primitive type, no nulls)

Returns:
    NumPy ndarray viewing the same memory

Raises:
    ValueError: If array has nulls or isn't contiguous
    TypeError: If type isn't supported for zero-copy

In [0]:
#| echo: false
#| output: asis
show_doc(arrow_to_max_tensor)

---

### arrow_to_max_tensor

```python

def arrow_to_max_tensor(
    arr:Union, device:Optional=None
)->Tensor:


```

*Zero-copy bridge from PyArrow array to MAX Tensor.*

Path: Arrow → NumPy view → MAX Tensor

On CPU: True zero-copy (same memory) via from_numpy
On GPU: Copies to device memory (unavoidable)

Args:
    arr: PyArrow Array or ChunkedArray
    device: Target device (None = CPU)

Returns:
    MAX Tensor ready for graph execution

Example:
    >>> arr = pa.array([1.0, 2.0, 3.0])
    >>> tensor = arrow_to_max_tensor(arr)
    >>> # tensor shares memory with arr on CPU

## 📊 MXFrame Class

A DataFrame wrapper that bridges PyArrow tables to MAX Engine.

### Design Principles

1. **PyArrow-native storage** - Uses Arrow's memory format internally
2. **Zero-copy by default** - `to_numpy()` and `to_max_tensor()` don't copy on CPU
3. **Cached views** - NumPy views are cached to keep memory references alive
4. **Device-aware** - Seamless CPU/GPU tensor creation

### API Overview

| Method | Returns | Zero-Copy? |
|--------|---------|------------|
| `to_numpy(col)` | `np.ndarray` | ✅ Yes |
| `to_max_tensor(col)` | `Tensor` (CPU) | ✅ Yes |
| `to_max_tensor(col, gpu)` | `Tensor` (GPU) | ❌ Copies to device |
| `get_buffer_address(col)` | `int` | N/A (for verification) |

In [0]:
#| echo: false
#| output: asis
show_doc(MXFrame)

---

### MXFrame

```python

def MXFrame(
    data:Union
):


```

*PyArrow-backed DataFrame with zero-copy MAX Engine integration.*

Stores data in PyArrow format and provides zero-copy access
to MAX tensors for GPU/CPU compute.

Example:
    >>> df = MXFrame({'price': [10.0, 20.0], 'qty': [1, 2]})
    >>> tensor = df.to_max_tensor('price')
    >>> # tensor shares memory with underlying Arrow buffer

## 🧪 Tests: Verify Zero-Copy

Critical tests to prove we're not copying data. These tests verify:

1. ✅ Memory addresses match between Arrow, NumPy, and MAX
2. ✅ Immutability is preserved (Arrow buffers are read-only)
3. ✅ Special types like Date32 work correctly
4. ✅ Performance matches zero-copy expectations (~4+ GB/s)
5. ✅ GPU transfer works when available
6. ✅ Proper error handling for unsupported cases

In [ ]:
import datetime
import time

In [ ]:
# 📋 Test 1: Create MXFrame
df = MXFrame({
    'price': [10.0, 20.0, 30.0, 40.0, 50.0],
    'qty': [1, 2, 3, 4, 5],
})

print(f"Created: {df}")
print(f"Columns: {df.columns}")
print(f"Rows: {df.num_rows}")

In [ ]:
# 🔍 Test 2: Zero-copy verification - Arrow to NumPy
arrow_arr = df['price']
numpy_view = df.to_numpy('price')

# Get addresses
arrow_addr = df.get_buffer_address('price')
numpy_addr = numpy_view.ctypes.data

print(f"Arrow buffer address:  0x{arrow_addr:X}")
print(f"NumPy array address:   0x{numpy_addr:X}")
print(f"Same memory: {arrow_addr == numpy_addr} ✓" if arrow_addr == numpy_addr else "COPY DETECTED!")

In [ ]:
# ⚡ Test 3: Arrow to MAX Tensor
tensor = df.to_max_tensor('price')

print(f"Tensor shape: {tensor.shape}")
print(f"Tensor dtype: {tensor.dtype}")
print(f"Tensor values: {tensor.to_numpy()}")
print(f"Arrow values:  {df['price'].to_pylist()}")

In [ ]:
# 🔒 Test 4: Verify NumPy view is read-only (Arrow buffers are immutable)
numpy_view = df.to_numpy('price')

try:
    numpy_view[0] = 999.0
    print("WARNING: NumPy view is writable")
except ValueError:
    print("NumPy view is read-only (expected - Arrow buffers are immutable) ✓")
    
# But they share memory - check via address
print(f"Same memory confirmed by address match: {df.get_buffer_address('price') == numpy_view.ctypes.data}")

In [ ]:
# 🔢 Test 5: Integer column
numpy_qty = df.to_numpy('qty')
tensor_qty = df.to_max_tensor('qty')

print(f"qty NumPy dtype: {numpy_qty.dtype}")
print(f"qty Tensor dtype: {tensor_qty.dtype}")
print(f"qty values: {numpy_qty}")

In [ ]:
# 📅 Test 6: Date32 column (special handling)


dates = pa.array([
    datetime.date(2024, 1, 1),
    datetime.date(2024, 6, 15),
    datetime.date(2024, 12, 31),
], type=pa.date32())

df_dates = MXFrame(pa.table({'shipdate': dates}))

numpy_dates = df_dates.to_numpy('shipdate')
print(f"Date32 as int32 (epoch days): {numpy_dates}")
print(f"dtype: {numpy_dates.dtype}")

In [ ]:
# 🚀 Test 7: Large array performance test (proves zero-copy)

n = 10_000_000
large_arr = pa.array(np.random.rand(n).astype(np.float32))
df_large = MXFrame(pa.table({'data': large_arr}))

# Time the zero-copy conversion
t0 = time.perf_counter()
for _ in range(100):
    tensor = df_large.to_max_tensor('data')
elapsed = (time.perf_counter() - t0) / 100 * 1000

print(f"Array size: {n:,} elements ({n * 4 / 1e6:.1f} MB)")
print(f"Arrow → MAX Tensor: {elapsed:.3f} ms")
print(f"Throughput: {n * 4 / 1e9 / (elapsed / 1000):.1f} GB/s")
print("(Fast time = zero-copy confirmed)")

In [ ]:
# 🎮 Test 8: GPU transfer (if available)
if driver.accelerator_count() > 0:
    gpu = driver.Accelerator()
    
    t0 = time.perf_counter()
    gpu_tensor = df_large.to_max_tensor('data', device=gpu)
    gpu_time = (time.perf_counter() - t0) * 1000
    
    print(f"CPU → GPU transfer: {gpu_time:.2f} ms")
    print(f"GPU tensor shape: {gpu_tensor.shape}")
else:
    print("No GPU available - skipping GPU test")

In [ ]:
# ❌ Test 9: Error handling - nulls should fail
arr_with_nulls = pa.array([1.0, None, 3.0])
df_nulls = MXFrame(pa.table({'col': arr_with_nulls}))

try:
    _ = df_nulls.to_numpy('col')
    print("ERROR: Should have raised ValueError!")
except ValueError as e:
    print(f"Correctly rejected nulls: {e}")

In [ ]:
print("\n" + "="*50)
print("✅ ALL ZERO-COPY BRIDGE TESTS PASSED")
print("="*50)